In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
import sys
sys.path.append("..")  # so the notebook can import from src/

import pandas as pd
import numpy as np
import lightgbm
import sklearn

from src import config
from src.evaluate import evaluate

print("All imports OK")
print("Looking for data in:", config.RAW_DIR)

All imports OK
Looking for data in: D:\Projects\Main Projects\fraud-detection\data\raw


# **Loading and Merging The Datasets**

In [3]:
from src.data import load_merged

df = load_merged(save=True)
print("Shape:", df.shape)
df.head()

Shape: (590540, 434)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [5]:
from src.data import load_processed
df = load_processed()
print(df.shape)
print(df.head())

(590540, 434)
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   

   card2  card3       card4  card5  ...                id_31  id_32  \
0    NaN  150.0    discover  142.0  ...                  NaN    NaN   
1  404.0  150.0  mastercard  102.0  ...                  NaN    NaN   
2  490.0  150.0        visa  166.0  ...                  NaN    NaN   
3  567.0  150.0  mastercard  117.0  ...                  NaN    NaN   
4  514.0  150.0  mastercard  102.0  ...  samsung browser 6.2   32.0   

       id_33           id_34  id_35 id_36 id_37  id_38  DeviceType  \
0        NaN             N

# **Data Exploration**

In [4]:
# Check for imbalance
print(df[config.TARGET].value_counts())
print(df[config.TARGET].value_counts(normalize=True))

isFraud
0    569877
1     20663
Name: count, dtype: int64
isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64


In [5]:
# Check for missing values
miss = df.isna().mean().sort_values(ascending=False)
print("Columns >90% missing:", (miss > 0.90).sum())
print("Columns >50% missing:", (miss > 0.50).sum())
miss.head(20)

Columns >90% missing: 12
Columns >50% missing: 214


id_24    0.991962
id_25    0.991310
id_07    0.991271
id_08    0.991271
id_21    0.991264
id_26    0.991257
id_27    0.991247
id_23    0.991247
id_22    0.991247
dist2    0.936284
D7       0.934099
id_18    0.923607
D13      0.895093
D14      0.894695
D12      0.890410
id_04    0.887689
id_03    0.887689
D6       0.876068
id_33    0.875895
id_09    0.873123
dtype: float64

In [6]:
# Columns information
groups = {}
for col in df.columns:
    prefix = ''.join([c for c in col if not c.isdigit()])
    groups.setdefault(prefix, []).append(col)
for prefix, cols in sorted(groups.items(), key=lambda x: -len(x[1]))[:15]:
    print(f"{prefix:15s} {len(cols)} cols")

V               339 cols
id_             38 cols
D               15 cols
C               14 cols
M               9 cols
card            6 cols
addr            2 cols
dist            2 cols
TransactionID   1 cols
isFraud         1 cols
TransactionDT   1 cols
TransactionAmt  1 cols
ProductCD       1 cols
P_emaildomain   1 cols
R_emaildomain   1 cols


In [7]:
# Transaction amount details
df['TransactionAmt'].describe()

count    590540.000000
mean        135.027176
std         239.162522
min           0.251000
25%          43.321000
50%          68.769000
75%         125.000000
max       31937.391000
Name: TransactionAmt, dtype: float64

In [8]:
# Transaction amount on the basis of fraud and non-fraud
df.groupby(config.TARGET)['TransactionAmt'].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,569877.0,134.511665,239.395078,0.251,43.970,68.5,120.0,31937.391
1,20663.0,149.244779,232.212163,0.292,35.044,75.0,161.0,5191.000


In [9]:
# Analyzing time column for split decision
print(df[config.TIME_COL].min(), df[config.TIME_COL].max())
span_days = (df[config.TIME_COL].max() - df[config.TIME_COL].min()) / (60*60*24)
print(f"Span: {span_days:.1f} days")

86400 15811131
Span: 182.0 days


In [10]:
# Categorical sanity checks
for col in ['ProductCD', 'card4', 'card6']:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))


ProductCD:
ProductCD
W    439670
C     68519
R     37699
H     33024
S     11628
Name: count, dtype: int64

card4:
card4
visa                384767
mastercard          189217
american express      8328
discover              6651
NaN                   1577
Name: count, dtype: int64

card6:
card6
debit              439938
credit             148986
NaN                  1571
debit or credit        30
charge card            15
Name: count, dtype: int64


In [11]:
# Fraud rate by category
df.groupby('card6')[config.TARGET].mean()

card6
charge card        0.000000
credit             0.066785
debit              0.024263
debit or credit    0.000000
Name: isFraud, dtype: float64